# 藏东南山区多源遥感影像年度合成与分块导出

（1）设置路径、年份、波段及导出参数，初始化 Earth Engine。  
（2）读取网格矢量及其投影、编号。  
（3）定义 QA 去云、反射率还原和季节中值合成函数。  
（4）读取研究区边界，筛选与研究区相交的网格。  
（5）预览合成影像。  
（6）提交指定年份的分块导出任务，输出到 Google Drive。


In [69]:
import ee
# 把 import geemap 改为：
import geemap
from IPython.display import display
import os
import geopandas as gpd
from shapely.geometry import mapping
from tqdm import tqdm
from pathlib import Path

## 1. 全局与藏东南区域配置

In [ ]:
# 初始化 Earth Engine
GEE_PROJECT = os.getenv("GEE_PROJECT", "ee-2119125351kfw")
try:
    ee.Initialize(project=GEE_PROJECT)
except Exception:
    print('[AUTH] 本地凭证失效或不匹配，正在请求重新授权...')
    ee.Authenticate(auth_mode='localhost', force=True)
    ee.Initialize(project=GEE_PROJECT)

# 参数设置
BOUNDARY_LOCAL_PATH = Path("")
TILES_LOCAL_PATH = Path("")  # 填入网格矢量路径。

TARGET_YEAR = 2020
START_YEAR = 2018
END_YEAR = 2022  # 包含起止年份。

EXPORT_FOLDER = "SETP_rsimg"
MAX_CLOUD_PERCENT = 10

REGION_NAME = "SETP"
EXPORT_SCALE = 30
DRIVE_FORMAT = "GeoTIFF"
PRIMARY_MONTH_START = 8
PRIMARY_MONTH_END = 9
EXT_MONTH_START = 6  
EXT_MONTH_END = 10

OPTICAL_BANDS = ["Blue", "Green", "Red", "NIR", "SWIR1", "SWIR2"]

COLLECTIONS = {
    # --- Tier 1 数据源 ---
    "L5_L2":    {"id": "LANDSAT/LT05/C02/T1_L2", "bands": ["SR_B1", "SR_B2", "SR_B3", "SR_B4", "SR_B5", "SR_B7"], "scale": 0.0000275, "offset": -0.2, "type": "landsat", "level": "L2", "start_year": 1984, "end_year": 2012},
    "L7_L2":    {"id": "LANDSAT/LE07/C02/T1_L2", "bands": ["SR_B1", "SR_B2", "SR_B3", "SR_B4", "SR_B5", "SR_B7"], "scale": 0.0000275, "offset": -0.2, "type": "landsat", "level": "L2", "start_year": 1999, "end_year": 2024},
    "L8_L2":    {"id": "LANDSAT/LC08/C02/T1_L2", "bands": ["SR_B2", "SR_B3", "SR_B4", "SR_B5", "SR_B6", "SR_B7"], "scale": 0.0000275, "offset": -0.2, "type": "landsat", "level": "L2", "start_year": 2013, "end_year": 9999},
    "L9_L2":    {"id": "LANDSAT/LC09/C02/T1_L2", "bands": ["SR_B2", "SR_B3", "SR_B4", "SR_B5", "SR_B6", "SR_B7"], "scale": 0.0000275, "offset": -0.2, "type": "landsat", "level": "L2", "start_year": 2021, "end_year": 9999},
    "S2_L2":    {"id": "COPERNICUS/S2_SR_HARMONIZED", "bands": ["B2", "B3", "B4", "B8", "B11", "B12"], "scale": 0.0001, "offset": 0.0, "type": "s2", "level": "L2", "start_year": 2017, "end_year": 9999},

}

## 2. 导入网格矢量文件

In [71]:
tiles_gdf = gpd.read_file(TILES_LOCAL_PATH)

grid_export_crs = tiles_gdf.crs
if grid_export_crs.is_geographic:
    grid_export_crs = tiles_gdf.estimate_utm_crs()
grid_export_crs = grid_export_crs.to_string()
tiles_gdf = tiles_gdf.to_crs('EPSG:4326')
print('导入网格数：', len(tiles_gdf))
if 'tile_id' not in tiles_gdf.columns:
    tiles_gdf['tile_id'] = range(1, len(tiles_gdf) + 1)
tiles_gdf['tile_id_num'] = tiles_gdf['tile_id'].astype(str).str.replace('Tile_', '', regex=False).astype(int)


导入网格数： 22


## 3. 影像合成

In [72]:
def preprocess_image(img, sensor):
    """单景影像清洗：云雪掩膜、异常剔除、物理反射率还原"""
    cfg = COLLECTIONS[sensor]

    # 质量控制与云雪掩膜
    if cfg["type"] == "landsat":
        qa_clear = img.select("QA_PIXEL").bitwiseAnd(31).eq(0)
        valid_rad = img.select("QA_RADSAT").eq(0)
        clear_mask = qa_clear.And(valid_rad)
    else:
        qa = img.select("QA60")
        qa_clear = qa.bitwiseAnd(1 << 10).eq(0).And(qa.bitwiseAnd(1 << 11).eq(0))
        scl = img.select("SCL")
        scl_clear = (scl.neq(0).And(scl.neq(1)).And(scl.neq(3))
                     .And(scl.neq(8)).And(scl.neq(9)).And(scl.neq(10)))
        clear_mask = qa_clear.unmask(scl_clear).And(scl_clear)

    # 提取有数据的有效范围
    has_data = img.select(cfg["bands"]).mask().reduce(ee.Reducer.min())
    img_masked = img.updateMask(clear_mask.And(has_data))

    # 物理反射率解压映射与波段统一
    scaled_img = (img_masked.select(cfg["bands"], OPTICAL_BANDS)
                  .multiply(cfg["scale"])
                  .add(cfg["offset"])
                  .toFloat())
    return scaled_img.set('system:time_start', img.get('system:time_start'))
def build_period_composite(start_year, end_year, region_geom, melt_start, melt_end, ext_start, ext_end, cloud_thresh):
    """跨传感器合成：核心消融期优先，边缘月份兜底，屏蔽冬季积雪"""
    processed_collections = []

    for sname, cfg in COLLECTIONS.items():
        if cfg["start_year"] > end_year or cfg["end_year"] < start_year:
            continue

        cloud_prop = "CLOUDY_PIXEL_PERCENTAGE" if cfg["type"] == "s2" else "CLOUD_COVER"
        
        col = (ee.ImageCollection(cfg["id"])
               .filterBounds(region_geom)
               .filter(ee.Filter.calendarRange(start_year, end_year, "year"))
               .filter(ee.Filter.calendarRange(ext_start, ext_end, "month")) 
               .filter(ee.Filter.gte(cloud_prop, 0))
               .filter(ee.Filter.lt(cloud_prop, cloud_thresh))
               .map(lambda img, s=sname: preprocess_image(img, s)))
        processed_collections.append(col)

    if not processed_collections:
        raise ValueError(f"在 {start_year}-{end_year} 期间未找到匹配的卫星数据！")

    merged_col = processed_collections[0]
    for c in processed_collections[1:]:
        merged_col = merged_col.merge(c)

    empty_img = (ee.Image.constant([0]*6)
                 .rename(OPTICAL_BANDS)
                 .toFloat()
                 .updateMask(ee.Image.constant(0))
                 .set('system:time_start', ee.Date.fromYMD(start_year, 1, 1).millis()))
    
    fallback_col = ee.ImageCollection([empty_img])

    primary_filter = ee.Filter.calendarRange(melt_start, melt_end, "month")
    shoulder_filter = ee.Filter.Not(primary_filter) 
    
    primary_median = merged_col.filter(primary_filter).merge(fallback_col).median()
    shoulder_median = merged_col.filter(shoulder_filter).merge(fallback_col).median()
    
    optical_mosaic = primary_median.unmask(shoulder_median, sameFootprint=False)

    encoded = optical_mosaic.multiply(10000).round()
    representable = encoded.gte(0).And(encoded.lte(65534)).reduce(ee.Reducer.min())
    
    final_output = (encoded.updateMask(representable)
                    .toUint16()
                    .unmask(65535, sameFootprint=False)
                    .clip(region_geom))
                    
    return final_output


## 4. 读取研究区边界并筛选相交网格

In [73]:
boundary_gdf = gpd.read_file(BOUNDARY_LOCAL_PATH)
boundary_wgs84 = boundary_gdf.to_crs("EPSG:4326")
boundary_union = boundary_wgs84.geometry.union_all()
valid_tiles = tiles_gdf[tiles_gdf.intersects(boundary_union)].copy()
if valid_tiles.empty:
    raise ValueError("网格与研究区没有交集。")
study_geom_ee = ee.Geometry(mapping(boundary_union))
print("研究区：", REGION_NAME, "相交网格数：", len(valid_tiles))


研究区： SETP 相交网格数： 22


## 5. 导出前预览

确认影像后，再运行后面的导出单元格。真彩色默认显示，假彩色和网格可在图层面板切换。

In [74]:
# 生成当前年份的预览影像；不提交导出任务。
study_area = ee.FeatureCollection([ee.Feature(study_geom_ee)])
selected_tiles = geemap.gdf_to_ee(valid_tiles)
final_img = build_period_composite(START_YEAR, END_YEAR, study_geom_ee,
                                   PRIMARY_MONTH_START, PRIMARY_MONTH_END,
                                   EXT_MONTH_START, EXT_MONTH_END, MAX_CLOUD_PERCENT)

preview_img = final_img.updateMask(final_img.select(0).neq(65535))

map = geemap.Map(height="650px")
map.centerObject(study_area, 7)
vis_true = {'bands': ['Red', 'Green', 'Blue'], 'min': 0, 'max': 3000}
vis_false = {'bands': ['SWIR1', 'NIR', 'Red'], 'min': 0, 'max': 5000}

map.addLayer(preview_img, vis_false, f'{TARGET_YEAR} 合成假彩色（{START_YEAR}—{END_YEAR}）', True)
map.addLayer(preview_img, vis_true, f'{TARGET_YEAR} 合成真彩色（{START_YEAR}—{END_YEAR}）', False)
map.addLayer(study_area.style(color='FFFFFF', fillColor='00000000', width=2), {}, '研究区边界')
# map.addLayer(selected_tiles.style(color='FF0000', fillColor='00000000', width=1), {}, '待导出网格', False)
map.add_layer_control()
display(map)


Map(center=[29.790744893784154, 95.21823162922048], controls=(WidgetControl(options=['position', 'transparent_…

## 6. 指定单个年份并提交导出任务

在参数区修改 START_YEAR 和 END_YEAR 后运行。本格仅提交任务，可在 Earth Engine Tasks 查看执行结果；重复运行会重新提交任务。

In [75]:
# # 每次只导出 TARGET_YEAR 指定的一年。
# print(f'\n--- 派发年代 {str(TARGET_YEAR)} ({TARGET_YEAR}-{TARGET_YEAR}) ---')
# for index_counter, (_, row) in tqdm(enumerate(valid_tiles.iterrows()), total=len(valid_tiles), desc=f'{REGION_NAME}-{str(TARGET_YEAR)}'):
#     tile_id = f"Tile_{int(row['tile_id_num']):04d}"
#     description = f'{REGION_NAME}_{TARGET_YEAR}_{START_YEAR}_{END_YEAR}_{tile_id}_30m'
#     tile_geom_ee = ee.Geometry(mapping(row.geometry)).intersection(study_geom_ee, 1)
#     try:
#         tile_crs = str(row["proj"]).strip() if "proj" in row.index else grid_export_crs
#         composite = build_period_composite(START_YEAR, END_YEAR, tile_geom_ee, PRIMARY_MONTH_START, PRIMARY_MONTH_END, EXT_MONTH_START, EXT_MONTH_END, MAX_CLOUD_PERCENT)
#         target_folder = EXPORT_FOLDER
#         task = ee.batch.Export.image.toDrive(image=composite, description=description, folder=target_folder, scale=EXPORT_SCALE, region=tile_geom_ee, crs=tile_crs, maxPixels=10000000000000.0, fileFormat=DRIVE_FORMAT, formatOptions={'cloudOptimized': True, 'noData': 65535})
#         task.start()
#     except Exception as e:
#         print(f'\n[ERROR] 派发失败 {description}: {e}')
